# Metering token usage and spendThe course keeps API costs to a few dollars, but it never shows you what anyindividual call actually costs. This notebook wires a meter into the sameclients used from week 1 onwards and reports tokens and estimated spend.The meter wraps `chat.completions.create`, so every provider the course drivesthrough the OpenAI client is covered by one wrapper.

In [ ]:
import osfrom dotenv import load_dotenvfrom openai import OpenAIfrom cost_meter import CostMeterload_dotenv(override=True)

## Wrapping a client`wrap` patches the client in place and hands it back, so the rest of your codeis unchanged.

In [ ]:
meter = CostMeter()openai = meter.wrap(OpenAI())question = [{"role": "user", "content": "In one sentence, what is a token?"}]response = openai.chat.completions.create(model="gpt-4.1-nano", messages=question)print(response.choices[0].message.content)

In [ ]:
meter.report()

## Comparing providersEvery client below is the same `OpenAI` class with a different `base_url`, whichis exactly how `week2/day1.ipynb` sets them up. Only the providers you have keysfor will run.

In [ ]:
endpoints = {    "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai/", "GOOGLE_API_KEY", "gemini-2.5-flash-lite"),    "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-120b"),    "ollama": ("http://localhost:11434/v1", None, "llama3.2"),}comparison = CostMeter()for name, (url, env_var, model) in endpoints.items():    key = os.getenv(env_var) if env_var else "ollama"    if not key:        print(f"skipping {name}, no key set")        continue    client = comparison.wrap(OpenAI(base_url=url, api_key=key))    try:        client.chat.completions.create(model=model, messages=question)        print(f"called {name}")    except Exception as e:        print(f"{name} failed: {type(e).__name__}")comparison.report()

## What reasoning effort costs`week2/day1.ipynb` varies `reasoning_effort` on the puzzle prompts. Reasoningtokens are billed as output tokens but never appear in the reply, so the pricedifference is invisible without metering. Setting `meter.label` tags the callsthat follow.

In [ ]:
puzzle = [{"role": "user", "content": "A bat and ball cost $1.10. The bat costs $1 more than the ball. What does the ball cost?"}]effort_meter = CostMeter()gpt5 = effort_meter.wrap(OpenAI())for effort in ["minimal", "low", "medium"]:    effort_meter.label = effort    gpt5.chat.completions.create(        model="gpt-5-nano", messages=puzzle, reasoning_effort=effort    )for row in effort_meter.rows():    print(f"{row['label']:>8}  reasoning tokens: {row['reasoning_tokens']:>5}  cost: ${row['cost']:.6f}")

## Per-call detail`rows()` yields plain dicts, so pandas can take it directly.

In [ ]:
import pandas as pdpd.DataFrame(effort_meter.rows())

## Notes- Token counts come from the provider and are exact. Costs are estimates from  the table in `cost_meter.py`; check it against current pricing pages.- Cached-input and batch discounts are not modelled, so a cached run costs less  than reported, never more.- Models served from localhost are counted as free.- A model with no entry in the price table is reported as `unpriced` rather than  silently counted as zero. Pass `CostMeter(prices={...})` to add your own rates.